# Решения: RFM и groupby

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден — положите slim CSV рядом с ноутбуком')


ORDERS_PATH = _find('orders_slim.csv')
CUSTOMERS_PATH = _find('customers_slim.csv')
PAYMENTS_PATH = _find('payments_slim.csv')

orders = pd.read_csv(ORDERS_PATH, parse_dates=['order_purchase_timestamp'])
if 'order_delivered_customer_date' in orders.columns:
    orders['order_delivered_customer_date'] = pd.to_datetime(
        orders['order_delivered_customer_date'], errors='coerce'
    )
customers = pd.read_csv(CUSTOMERS_PATH)
payments = pd.read_csv(PAYMENTS_PATH)


## Урок. 1-5

In [ ]:
orders_pay = orders.merge(payments, on='order_id', how='left')
ref_date = orders['order_purchase_timestamp'].max()
rfm = (
    orders_pay.groupby('customer_id')
    .agg(
        last_purchase=('order_purchase_timestamp', 'max'),
        Frequency=('order_id', 'nunique'),
        Monetary=('payment_value', 'sum'),
    )
    .reset_index()
)
rfm['Recency'] = (ref_date - rfm['last_purchase']).dt.days
rfm = rfm[['customer_id', 'Recency', 'Frequency', 'Monetary']]
WHEN_RECENCY = (
    'Recency всегда считается относительно одной опорной даты для всей таблицы, '
    'иначе значения между клиентами не сопоставимы.'
)
stats = rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2)
f_mean = float(rfm['Frequency'].mean())
m_mean = float(rfm['Monetary'].mean())
top5 = rfm.sort_values('Monetary', ascending=False).head(5)
print(ref_date)
print(stats)
print('F mean:', round(f_mean, 2), 'M mean:', round(m_mean, 2))
print(top5)

## ДЗ. 1-3

In [ ]:
orders_pay = orders.merge(payments, on='order_id', how='left')
ref_date = orders['order_purchase_timestamp'].max()
rfm = (
    orders_pay.groupby('customer_id')
    .agg(last_purchase=('order_purchase_timestamp', 'max'), Frequency=('order_id', 'nunique'), Monetary=('payment_value', 'sum'))
    .reset_index()
)
rfm['Recency'] = (ref_date - rfm['last_purchase']).dt.days
rfm = rfm[['customer_id', 'Recency', 'Frequency', 'Monetary']]
rfm['freq_bin'] = pd.cut(rfm['Frequency'], bins=[0, 2, 5, 100], labels=['1-2', '3-5', '6+'])
freq_bin_share = rfm['freq_bin'].value_counts(normalize=True).sort_index()
top_recency = rfm.sort_values('Recency', ascending=False).head(10)
RFM_NOTE = (
    'Клиент с высоким Monetary и низким Frequency обычно делает редкие, но крупные покупки. '
    'Ему может подойти отдельная коммуникация: не частые скидки, а персональные дорогие предложения.'
)
print(freq_bin_share)
print(top_recency[['customer_id', 'Recency']])
print(RFM_NOTE)